In [1]:
# 根目录：里面可能有很多 run 子目录；每个子目录都含 frames/ 与 metadata.json
ROOT = "/data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006"
OVERWRITE = False        # 已存在是否覆盖
DRY_RUN = False          # 只打印将要做什么，不实际生成

In [2]:
import os, json, traceback
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib import gridspec
from pathlib import Path

def _nice_lim(vals, margin=1.0):
    lo, hi = float(np.min(vals)), float(np.max(vals))
    if np.isclose(lo, hi):
        lo, hi = lo - margin, hi + margin
    else:
        lo, hi = lo - margin, hi + margin
    return lo, hi

def _draw_text(img_arr, text):
    img = Image.fromarray(img_arr)
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), text, fill=(255, 0, 0))
    return np.array(img)

def _read_metadata(meta_path: str):
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta

In [3]:
def load_run(run_dir: str, add_text=True):
    """
    run_dir: 目录中应包含 metadata.json 和 frames/xxx.png
    返回: video[np.uint8 T,H,W,3], positions[List[(x,y,z,theta)]], meta(dict)
    """
    run_dir = Path(run_dir)
    meta_path = run_dir / "metadata.json"
    assert meta_path.exists(), f"找不到 {meta_path}"

    meta = _read_metadata(str(meta_path))
    frames_info = meta["frames"]
    fps = int(meta.get("fps", 2))

    video = []
    positions = []
    for fobj in frames_info:
        png_rel = fobj["png"]  # 可能是 "frames/xxx.png" 或仅文件名
        png_path = run_dir / Path(png_rel).name
        if not png_path.exists():
            # 兼容路径含 frames 子目录
            png_path = run_dir / png_rel
        assert png_path.exists(), f"帧不存在: {png_path}"

        img = np.array(Image.open(png_path).convert("RGB"))

        if add_text:
            step_idx = fobj["frame"]
            action   = fobj.get("action", "")
            img = _draw_text(img, f"Step {step_idx}: {action}")

        video.append(img)

        pose = fobj["pose"]
        positions.append((pose["x"], pose["y"], pose["z"], pose["theta_rad"]))

    # 统一尺寸
    H0, W0 = video[0].shape[:2]
    for i in range(len(video)):
        if video[i].shape[:2] != (H0, W0):
            video[i] = np.array(Image.fromarray(video[i]).resize((W0, H0)))

    video = np.stack(video, axis=0)
    meta["_fps"] = fps
    return video, positions, meta

In [4]:
def render_run_to_mp4(run_dir: str, save_name="output.mp4", overwrite=False, dry_run=False):
    """
    读取 run_dir 下的数据并保存 mp4 到 run_dir/save_name
    返回输出路径或 None
    """
    run_dir = Path(run_dir)
    out_mp4 = run_dir / save_name
    if out_mp4.exists() and not overwrite:
        print(f"[Skip] {out_mp4} 已存在（设置 OVERWRITE=True 可覆盖）")
        return str(out_mp4)

    if dry_run:
        print(f"[DryRun] 预备生成: {out_mp4}")
        return None

    video, positions, meta = load_run(str(run_dir))
    fps = int(meta.get("_fps", meta.get("fps", 2)))

    xs = [p[0] for p in positions]
    ys = [p[1] for p in positions]
    zs = [p[2] for p in positions]
    thetas = [p[3] for p in positions]

    fig = plt.figure(figsize=(12, 6))
    fig.suptitle(f"Run {meta.get('run_index', '')} | type={meta.get('type', '')}", fontsize=12)
    plt.subplots_adjust(top=0.9)

    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.5])
    ax1 = fig.add_subplot(gs[0])
    im1 = ax1.imshow(video[0])
    ax1.axis('off')
    ax1.set_title("Generated View")

    ax2 = fig.add_subplot(gs[1], projection='3d')
    ax2.set_title("3D Trajectory")
    ax2.set_xlim(*_nice_lim(xs, 1))
    ax2.set_ylim(*_nice_lim(ys, 1))
    ax2.set_zlim(*_nice_lim(zs, 1))

    traj_line, = ax2.plot([], [], [], 'bo-', lw=2)
    arrow = None

    def init():
        nonlocal arrow
        im1.set_data(video[0])
        traj_line.set_data([], [])
        traj_line.set_3d_properties([])
        if arrow is not None:
            try: arrow.remove()
            except Exception: pass
        dx, dy, dz = np.cos(thetas[0]), np.sin(thetas[0]), 0
        arrow = ax2.quiver(xs[0], ys[0], zs[0], dx, dy, dz, length=1.0, normalize=True, color='red')
        return im1, traj_line, arrow

    def animate(i):
        nonlocal arrow
        im1.set_data(video[i])
        traj_line.set_data(xs[:i+1], ys[:i+1])
        traj_line.set_3d_properties(zs[:i+1])
        if arrow is not None:
            try: arrow.remove()
            except Exception: pass
        dx, dy, dz = np.cos(thetas[i]), np.sin(thetas[i]), 0
        arrow = ax2.quiver(xs[i], ys[i], zs[i], dx, dy, dz, length=1.0, normalize=True, color='red')
        return im1, traj_line, arrow

    anim = animation.FuncAnimation(
        fig, animate, init_func=init,
        frames=min(len(video), len(positions)),
        interval=max(1, int(1000 // max(1, fps))),
        blit=False
    )

    anim.save(str(out_mp4), writer='ffmpeg', fps=fps)
    plt.close(fig)
    print(f"[OK] Saved: {out_mp4}")
    return str(out_mp4)

In [5]:
def iter_run_dirs(root: str):
    """
    递归找到所有包含 metadata.json 的目录（允许该目录名是 frames/ 或上层 run 目录）
    返回去重后的目录列表（即 metadata.json 所在目录）
    """
    root = Path(root)
    found = set()
    for p in root.rglob("metadata.json"):
        found.add(str(p.parent))
    return sorted(found)

def batch_render(root: str, save_name="output.mp4", overwrite=False, dry_run=False):
    run_dirs = iter_run_dirs(root)
    print(f"发现 {len(run_dirs)} 个 run 目录")
    outputs = []
    for rd in run_dirs:
        try:
            out = render_run_to_mp4(rd, save_name=save_name, overwrite=overwrite, dry_run=dry_run)
            outputs.append((rd, out, "ok"))
        except Exception as e:
            print(f"[ERR] {rd}\n{e}")
            traceback.print_exc()
            outputs.append((rd, None, "error"))
    return outputs

In [6]:
results = batch_render(ROOT)
print("完成，统计：")
ok = sum(1 for _,_,s in results if s=="ok")
err = sum(1 for _,_,s in results if s=="error")
print(f"OK: {ok} | ERR: {err}")
for rd, out, st in results:
    print(st, "->", rd, "=>", out)

发现 11 个 run 目录
[Skip] /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/output.mp4 已存在（设置 OVERWRITE=True 可覆盖）


[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_000/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_001/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_002/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_003/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_004/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_005/output.mp4
[OK] Saved: /data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006/candidates/cand_006/output.mp4
[OK] Saved: /